In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import xgboost as xgb



In [2]:
train_data = pd.read_csv("../input/ventilator-pressure-prediction/train.csv")
test_data = pd.read_csv("../input/ventilator-pressure-prediction/test.csv")
test_ids = test_data["id"].tolist()



In [3]:
missing_values_count = train_data.isnull().sum()
print(missing_values_count)
total_cells = np.product(train_data.shape)
total_missing = missing_values_count.sum()
print("% of missing data =", (total_missing / total_cells) * 100)




id           0
breath_id    0
R            0
C            0
time_step    0
u_in         0
u_out        0
pressure     0
dtype: int64
% of missing data = 0.0


In [4]:
def add_features(df):
    df["area"] = df["time_step"] * df["u_in"]
    df["area"] = df.groupby("breath_id")["area"].cumsum()

    df["u_in_cumsum"] = df["u_in"].groupby(df["breath_id"]).cumsum()

    df["u_in_lag1"] = df.groupby("breath_id")["u_in"].shift(1)
    df["u_out_lag1"] = df.groupby("breath_id")["u_out"].shift(1)
    df["u_in_lag_back1"] = df.groupby("breath_id")["u_in"].shift(-1)
    df["u_out_lag_back1"] = df.groupby("breath_id")["u_out"].shift(-1)
    df["u_in_lag2"] = df.groupby("breath_id")["u_in"].shift(2)
    df["u_out_lag2"] = df.groupby("breath_id")["u_out"].shift(2)
    df["u_in_lag_back2"] = df.groupby("breath_id")["u_in"].shift(-2)
    df["u_out_lag_back2"] = df.groupby("breath_id")["u_out"].shift(-2)
    df["u_in_lag3"] = df.groupby("breath_id")["u_in"].shift(3)
    df["u_out_lag3"] = df.groupby("breath_id")["u_out"].shift(3)
    df["u_in_lag_back3"] = df.groupby("breath_id")["u_in"].shift(-3)
    df["u_out_lag_back3"] = df.groupby("breath_id")["u_out"].shift(-3)
    df["u_in_lag4"] = df.groupby("breath_id")["u_in"].shift(4)
    df["u_out_lag4"] = df.groupby("breath_id")["u_out"].shift(4)
    df["u_in_lag_back4"] = df.groupby("breath_id")["u_in"].shift(-4)
    df["u_out_lag_back4"] = df.groupby("breath_id")["u_out"].shift(-4)
    df = df.fillna(0)

    df["breath_id__u_in__max"] = df.groupby("breath_id")["u_in"].transform("max")
    df["breath_id__u_out__max"] = df.groupby("breath_id")["u_out"].transform("max")

    df["u_in_diff1"] = df["u_in"] - df["u_in_lag1"]
    df["u_out_diff1"] = df["u_out"] - df["u_out_lag1"]
    df["u_in_diff2"] = df["u_in"] - df["u_in_lag2"]
    df["u_out_diff2"] = df["u_out"] - df["u_out_lag2"]

    df["breath_id__u_in__diffmax"] = (
        df.groupby("breath_id")["u_in"].transform("max") - df["u_in"]
    )
    df["breath_id__u_in__diffmean"] = (
        df.groupby("breath_id")["u_in"].transform("mean") - df["u_in"]
    )

    df["u_in_diff3"] = df["u_in"] - df["u_in_lag3"]
    df["u_out_diff3"] = df["u_out"] - df["u_out_lag3"]
    df["u_in_diff4"] = df["u_in"] - df["u_in_lag4"]
    df["u_out_diff4"] = df["u_out"] - df["u_out_lag4"]
    df["cross"] = df["u_in"] * df["u_out"]
    df["cross2"] = df["time_step"] * df["u_out"]

    df["R"] = df["R"].astype(str)
    df["C"] = df["C"].astype(str)
    df["R__C"] = df["R"] + "__" + df["C"]
    df = pd.get_dummies(df)
    return df




In [5]:
# Feature engineering for both splits
train_data = add_features(train_data)
test_data = add_features(test_data)

# Remove ID columns from the model inputs
train_data.drop(["id", "breath_id"], axis=1, inplace=True)
test_data.drop(["id", "breath_id"], axis=1, inplace=True)

# Align test columns to train columns (excluding the target)
train_feature_cols = [c for c in train_data.columns if c != "pressure"]
test_data = test_data.reindex(columns=train_feature_cols, fill_value=0)



In [6]:
# Split into training / validation sets
X = train_data[train_feature_cols]
y = train_data["pressure"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=0)

# Define a CPU‑compatible XGBoost regressor
regressor = xgb.XGBRegressor(
    tree_method="hist",
    colsample_bytree=0.9,
    alpha=0.01563,
    gamma=0.0,
    learning_rate=0.5,
    max_depth=15,
    min_child_weight=257,
    n_estimators=1024,
    reg_alpha=0.9,
    reg_lambda=0.003,
    subsample=0.9,
    random_state=2020,
)

# Fit with early stopping
regressor.fit(
    X_train, y_train, early_stopping_rounds=6, eval_set=[(X_val, y_val)], verbose=False
)



/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


XGBRegressor(alpha=0.01563, base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.0, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.5, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=15, max_leaves=None,
             min_child_weight=257, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1024, n_jobs=None,
             num_parallel_tree=None, ...)

In [7]:
# Predict on the test set and write submission
predictions = regressor.predict(test_data)
output = pd.DataFrame({"id": test_ids, "pressure": predictions})
output.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")



Submission saved to submission.csv


In [8]:
output.head()

,id,pressure
0,1,6.488458
1,2,6.467021
2,3,6.738573
3,4,8.561619
4,5,10.220480
